# Valo Upscale — thư mục công khai (UI trên Kaggle / Colab)

Nhập **một link công khai** (thư mục chứa video/ảnh) → notebook tải về → upscale bằng GPU → đẩy kết quả
lên nơi bạn chọn.

**Chuẩn bị:**
* Kaggle: *Settings → Accelerator = **GPU T4 x2***, *Internet = **On***.
* Colab: *Runtime → Change runtime type → **T4 GPU***.

⚠️ **Sự thật cần biết:** link công khai **chỉ đọc được**. Muốn đẩy kết quả *trở lại đúng thư mục đó* thì
phải có quyền ghi: Google Drive → Service Account JSON; Dropbox → access token; GitHub → PAT.
Không có credential thì chọn `kaggle_local` rồi tải file ở tab **Output** (sau khi Save version).

In [ ]:
#@title 1) Clone repo + cài thư viện
REPO_URL = "https://github.com/hoannguyen171/UPSCALEVIDEO.git"  #@param {type:"string"}
REPO_DIR = "/kaggle/working/valo-upscale-gpu"  #@param {type:"string"}

import os, shutil, subprocess

if not shutil.which("ffmpeg"):
    !apt-get -qq install -y ffmpeg

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True, text=True).stdout)
else:
    !git clone --depth 1 "{REPO_URL}" "{REPO_DIR}"

%cd {REPO_DIR}
!pip install -q -r requirements.txt -r requirements-remote.txt
!git log --oneline -1

In [ ]:
#@title 2) Kiểm tra môi trường
import shutil, torch, gdown
import valo_upscale
import valo_upscale.remote as remote

print("valo-upscale-gpu", valo_upscale.__version__, "| gdown", gdown.__version__)
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"))
print("ffmpeg:", shutil.which("ffmpeg"))


## 3) Xem trước nội dung thư mục (không tải)

Dán link vào và chạy — hữu ích để biết link có đọc được không, có bao nhiêu file media.
Hỗ trợ: Google Drive (folder/ file công khai), GitHub (thư mục repo), Dropbox (cần token),
hoặc trang index HTTP / danh sách URL.

In [ ]:
#@title 3) Liệt kê file trong link
FOLDER_URL = ""  #@param {type:"string"}

if FOLDER_URL.strip():
    !python -m valo_upscale.kaggle_app --list --url "{FOLDER_URL}"
else:
    print("Hãy dán link vào FOLDER_URL ở trên (ví dụ: https://drive.google.com/drive/folders/XXXX)")


## 4) Mở giao diện

Bấm chạy ô dưới, đợi dòng `Running on public URL: https://....gradio.live` rồi mở link đó.
Trong UI: dán link thư mục → chọn scale/model → chọn đích đẩy kết quả → bấm **Tải → Upscale → Đẩy lên**.
Ô này chạy mãi cho tới khi bạn bấm **Stop**.

In [ ]:
#@title 4) Mở UI Gradio (link *.gradio.live)
!pip install -q -r requirements-ui.txt

import valo_upscale.kaggle_app as ka, os
work = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
demo = ka.build_demo(work_dir=work)
demo.queue().launch(share=True, debug=False)

## 5) Chạy không cần giao diện (headless)

Dùng khi bạn muốn chạy tự động. Token đọc từ biến môi trường (đừng gõ token vào lệnh):

In [ ]:
#@title 5) Chạy headless
FOLDER_URL = ""  #@param {type:"string"}
SCALE = 4  #@param [2, 3, 4, 6, 8] {type:"raw"}
LIMIT = 0  #@param {type:"integer"}
UPLOAD_KIND = "kaggle_local"  #@param ["kaggle_local", "gdrive", "dropbox", "github"]
UPLOAD_FOLDER = ""  #@param {type:"string"}

import os
os.environ["DROPBOX_TOKEN"] = os.environ.get("DROPBOX_TOKEN", "")
os.environ["GITHUB_TOKEN"] = os.environ.get("GITHUB_TOKEN", "")
# os.environ["GDRIVE_SA_JSON"] = "/kaggle/input/<dataset>/sa.json"   # nếu đẩy lên Drive

if FOLDER_URL.strip():
    import subprocess
    subprocess.run(["python", "-m", "valo_upscale.kaggle_app", "--headless", "--url", FOLDER_URL,
                    "--scale", str(SCALE), "--limit", str(LIMIT),
                    "--upload-kind", UPLOAD_KIND, "--upload-folder", UPLOAD_FOLDER], check=False)
else:
    print("Chưa có FOLDER_URL.")

## 6) Xem & lấy kết quả

* Kết quả nằm ở `/kaggle/working/upscaled` (Kaggle) — bấm **Save version** để lưu, tải ở tab **Output**.
* Colab: nén rồi tải về bằng `files.download`.
* Weight model nằm ở `~/.cache/valo-upscale/weights`; muốn giữ giữa các phiên thì set
  `VALO_WEIGHTS_DIR=/kaggle/working/weights` **trước** khi chạy.

In [ ]:
#@title 6) Liệt kê + nén kết quả
from pathlib import Path
import shutil, os

out = Path("/kaggle/working/upscaled") if Path("/kaggle/working/upscaled").exists() else Path("upscaled")
files = sorted(out.glob("*")) if out.exists() else []
print(f"{out}: {len(files)} file")
for f in files[:20]:
    print(f"  • {f.name} — {f.stat().st_size / 1048576:.1f} MB")

if files:
    zip_path = shutil.make_archive("/kaggle/working/ket_qua", "zip", root_dir=str(out))
    print("ZIP:", zip_path)
    if not Path("/kaggle/working").exists():          # Colab: tải trực tiếp
        from google.colab import files as colab_files
        colab_files.download(zip_path)

## 7) Lấy credential (chỉ khi đẩy lên Drive / Dropbox / GitHub)

| Đích | Cần gì | Lấy ở đâu |
|---|---|---|
| **Google Drive** | Service Account JSON, share thư mục đích cho `client_email` trong file JSON (quyền **Editor**), bật **Google Drive API** | console.cloud.google.com → *IAM & Admin → Service Accounts → Keys → Add key → JSON* |
| **Dropbox** | Access token (scope `files.content.read` + `files.content.write`); thư mục đích phải nằm trong Dropbox của bạn (bấm *Add to your Dropbox* nếu là thư mục được chia sẻ) | dropbox.com/developers/apps → *Create app* → *Generated access token* |
| **GitHub** | PAT (classic scope `repo`, hoặc fine-grained **Contents: Read and write**) và thư mục đích dạng `owner/repo/duong/dan` | github.com/settings/tokens |

Giới hạn cần nhớ: GitHub **100 MB/file**, Dropbox upload đơn **150 MB**, Drive thì thoải mái (nhưng
Service Account có hạn mức 15 GB dung lượng riêng).

Điền credential trong UI ở mục **Credential** (hoặc biến môi trường `GDRIVE_SA_JSON`, `DROPBOX_TOKEN`,
`GITHUB_TOKEN` khi chạy headless).